In [107]:
import requests
from bs4 import BeautifulSoup
import DB
import pandas as pd
import json
from tqdm import tqdm

In [175]:
matchlist = []
for tier in ['diamond', 'emerald', 'platinum', 'gold']:
    pbar = tqdm(range(1, 40),
               total=40,
               desc=f'Collecting Id at {tier}',
               leave=True)
    for page in pbar:
        url=f'https://op.gg/leaderboards/tier?tier={tier}&page={page}&region=kr'
        response=requests.get(url)
        html = response.text
        soup = BeautifulSoup(html, 'html.parser')
        page = soup.select('#content-container > div > div.flex.flex-col.gap-2.md\\:flex-row > main > table > tbody > tr')
        for summoner in page:
            idtag = summoner.get('id')
            if idtag:
                matchlist.append(idtag.split('-'))
        time.sleep(0.2)





ollecting Id at gold:  98%|█████████████████████████████████████████████████████████▌ | 39/40 [00:29<00:00,  1.32it/s]

In [176]:
matchlist_df = pd.DataFrame(matchlist)
matchlist_df.to_csv('./idlist.csv')

In [178]:
duolist = []
pbar = tqdm(matchlist,
           total=len(matchlist),
           desc='Crawling Duo IDs')
for player in pbar:
    url=f'https://api.your.gg/kr/api/profile/by-riot-id/{player[0]}-{player[1]}?lang=ko&matchCategory=SoloRank&listMatchCategory=SoloRank'
    try:
        response=requests.get(url, timeout=5)
    except:
        continue
    
    try:
        data = response.json()['duoList']
    except:
        continue
    
    for duo in data:
        if duo['lose'] + duo['win'] < 10:
            continue
        else:
            try:
                duolist.append({'p1GameName': player[0], 'p1Tagline': player[1], 'p2GameName': duo['riotIdGameName'], 'p2Tagline': duo['riotIdTagline'], 'winrate': duo['winRate']})
            except Exception as e:
                print(e)
                continue
    time.sleep(0.2)
    pbar.set_description(f'Crawling Duo IDs; size={len(duolist)}')


rawling Duo IDs; size=5145: 100%|█████████████████████████████████████████████| 15478/15478 [9:14:03<00:00,  2.15s/it]

In [179]:
duolist_df = pd.DataFrame(duolist)
duolist_df.to_csv('./duolist.csv')

# To 태호

In [ ]:
#csv 읽어오기

In [136]:
for duo in duolist: # 전체로 변경 필요
    url=f'https://api.your.gg/kr/api/summoners/by-riot-id/{duo['p1GameName']}-{duo['p1Tagline']}?lang=ko&matchCategory=SoloRank'
    response=requests.get(url)
    p1data=response.json()
    
    url=f'https://api.your.gg/kr/api/summoners/by-riot-id/{duo['p2GameName']}-{duo['p2Tagline']}?lang=ko&matchCategory=SoloRank'
    response=requests.get(url)
    p2data=response.json()
    
    

In [135]:
p1data

{'championVideo': 'https://youtube.com/embed/4KxkX3nc_O0',
 'competitor': {'name': 'Adc', 'tier': 'Master', 'type': 'Lane'},
 'division': '',
 'icon': 'https://ddragon.leagueoflegends.com/cdn/15.6.1/img/profileicon/5990.png',
 'kda': 4.074333333333334,
 'laning': 4.758448613616499,
 'lose': 27,
 'lp': 0,
 'matchCategory': 'SoloRank',
 'mostChampions': [{'id': 51,
   'imageUrl': 'https://ddragon.leagueoflegends.com/cdn/15.6.1/img/champion/Caitlyn.png',
   'isAverage': False,
   'kda': 4.63909090909091,
   'key': 'Caitlyn',
   'lane': 'Adc',
   'laning': 5.000588635268271,
   'matchCount': 11,
   'name': '케이틀린',
   'performance': {'assists': 7.2727272727272725,
    'assistsPerMinute': 0.2711903409901186,
    'carry': 4.779176860099089,
    'csPerMinute': 7.1302228129675305,
    'csdAt15': -5.2727272727272725,
    'cwPerMinute': 0.045858581537113245,
    'damagePerDeath': 6018.476515151516,
    'damagePerGold': 1.5010194770754346,
    'damagePerMinute': 649.2847184517374,
    'damagePerMi